# CAS Common Chemistry batch lookup

Looks up every CAS number in a CSV you upload against the CAS Common
Chemistry `/detail` API (https://commonchemistry.cas.org/api-overview)
and writes back one row per CAS number with whatever fields the API
returns: name, synonyms, molecular formula, molecular mass, SMILES,
InChI, InChIKey, replaced (superseded) CAS numbers, and the full raw
JSON response as a safety net.

**Before running:**

1. This was built and syntax-checked, but **never actually run against
   the live API** -- this sandbox's network policy blocks
   `commonchemistry.cas.org` entirely (confirmed directly), so I could
   not make a single real test call. The field names below
   (`rn`, `name`, `synonyms`, `molecularFormula`, `molecularMass`,
   `smile`, `canonicalSmile`, `inchi`, `inchiKey`, `replacedRns`,
   `hasMolfile`, `experimentalProperties`) are what CAS documents, not
   guesses, but **run the single-CAS test in section 2 first** and
   compare the printed raw JSON against what's extracted before you
   commit to a 14,072-row run. If a field name is off, it'll still show
   up in `raw_json` even if the dedicated column comes back blank --
   fix the field name in `FIELD EXTRACTORS` (section 3) and re-run.
2. **Coverage caveat**: Common Chemistry is a curated subset of the full
   CAS Registry, not all 200M+ registered substances -- expect a real
   fraction of 14,072 arbitrary CAS numbers to come back "not found."
   That's normal, not a bug; they're reported separately at the end.
3. **Rate limit**: I don't have a confirmed number for your key's rate
   limit, so this defaults to a conservative pace (`REQUESTS_PER_SECOND`
   below) and backs off automatically on HTTP 429. Raise the rate once
   you've confirmed it's not getting throttled, to cut down the total
   run time -- at 14,072 lookups even 3-4 requests/second is close to an
   hour.
4. The loop **checkpoints to disk every 50 rows** and skips CAS numbers
   already present in the output file, so a Colab disconnect partway
   through doesn't cost you the whole run -- just re-run the batch cell.
5. Your API key is requested via a hidden prompt (`getpass`) each run
   rather than typed into a cell, so it's never saved into the notebook
   file itself. Don't paste it directly into a code cell if you plan to
   share this notebook.


## 1. Setup

In [ ]:
import time
import json
import getpass
import requests
import pandas as pd
from pathlib import Path

API_BASE = "https://commonchemistry.cas.org/api"
CAS_API_KEY = getpass.getpass("Paste your CAS Common Chemistry API key (hidden): ")

SESSION = requests.Session()
SESSION.headers.update({"X-API-KEY": CAS_API_KEY, "Accept": "application/json"})


## 2. Test with a single, known CAS number first

Run this before anything else. `7732-18-5` is water -- if this doesn't
come back looking right (or 404s), something about the endpoint/header/
key is off and the batch run below will just burn through 14,072 failed
requests. Compare the printed fields against `raw_json` to catch any
field-name mismatch now rather than after a long run.

In [ ]:
def fetch_detail_raw(cas_rn, timeout=15):
    """One raw API call. Returns (status_code, json_or_None)."""
    resp = SESSION.get(f"{API_BASE}/detail", params={"cas_rn": cas_rn}, timeout=timeout)
    try:
        body = resp.json()
    except ValueError:
        body = None
    return resp.status_code, body

status, body = fetch_detail_raw("7732-18-5")
print("HTTP status:", status)
print(json.dumps(body, indent=2))


## 3. Field extraction

Defensive: every field is pulled with `.get()` so a missing/renamed
field just comes back blank rather than crashing the whole batch. The
full raw JSON is always kept too (`raw_json` column) so nothing the API
actually returned is lost even if a specific column below is wrong.

In [ ]:
def join_list(value, sep="; "):
    if not value:
        return ""
    if isinstance(value, list):
        # each item may be a plain string or a dict (e.g. experimentalProperties) --
        # stringify dicts rather than crash on them.
        return sep.join(v if isinstance(v, str) else json.dumps(v) for v in value)
    return str(value)

def extract_fields(cas_rn, status, body):
    row = {"cas_number_queried": cas_rn, "http_status": status}
    if status == 404 or body is None:
        row["found"] = False
        row["raw_json"] = "" if body is None else json.dumps(body)
        return row

    row["found"] = True
    row["rn"] = body.get("rn", "")
    row["name"] = body.get("name", "")
    row["synonyms"] = join_list(body.get("synonyms"))
    row["molecular_formula"] = body.get("molecularFormula", "")
    row["molecular_mass"] = body.get("molecularMass", "")
    row["smile"] = body.get("smile", "") or body.get("canonicalSmile", "")
    row["canonical_smile"] = body.get("canonicalSmile", "")
    row["inchi"] = body.get("inchi", "")
    row["inchi_key"] = body.get("inchiKey", "")
    row["replaced_rns"] = join_list(body.get("replacedRns"))
    row["has_molfile"] = body.get("hasMolfile", "")
    row["experimental_properties"] = join_list(body.get("experimentalProperties"))
    row["raw_json"] = json.dumps(body)
    return row

# sanity check against the water lookup from section 2
print(extract_fields("7732-18-5", status, body))


## 4. Upload your CAS number list

Upload a CSV containing your 14,072 CAS numbers, then set
`CAS_COLUMN` to the name of the column that holds them.

In [ ]:
from google.colab import files

uploaded = files.upload()
input_filename = next(iter(uploaded))
input_df = pd.read_csv(input_filename, dtype=str)
print(f"Loaded {len(input_df)} rows from {input_filename}")
print("Columns:", list(input_df.columns))
input_df.head()


In [ ]:
CAS_COLUMN = "cas_number"  # <-- set this to your actual CAS-number column name

assert CAS_COLUMN in input_df.columns, f"{CAS_COLUMN!r} not in {list(input_df.columns)}"
cas_numbers = (
    input_df[CAS_COLUMN]
    .dropna()
    .astype(str)
    .str.strip()
)
cas_numbers = cas_numbers[cas_numbers != ""].unique().tolist()
print(f"{len(cas_numbers)} unique, non-blank CAS numbers to look up")


## 4b. Resume safely across runtime restarts (recommended for a 14k-row run)

Colab's local disk (where `cas_common_chemistry_results.csv` normally lives)
is wiped every time the runtime restarts or disconnects -- which is why the
progress you'd already made disappeared and `load_existing_results()` in
Section 5 started over from zero, even though you had downloaded a copy to
your own computer.

Run this cell once per session. It mounts your Google Drive and points
`OUTPUT_PATH` at a file inside it instead of local disk, so:
- every checkpoint in Section 5 is saved to Drive, not just local disk
- if the runtime restarts, just re-run Setup (Section 1) then THIS cell --
  your results file is still sitting in Drive, `load_existing_results()`
  picks it up automatically, and Section 5 resumes where it left off. No
  more manual re-uploading.

**First time only:** if you already have a partial results CSV downloaded
locally (e.g. the ~10,000-row file from before), this cell will offer to
upload it once to seed Drive with that existing progress. After that first
seed, you never need to upload it again.

In [ ]:
import shutil
from pathlib import Path
from google.colab import drive, files

# if Drive is already mounted from an earlier cell run this session, drive.mount()
# just prints a notice and no-ops -- that message is expected, not an error.
drive.mount("/content/drive")

DRIVE_DIR = Path("/content/drive/MyDrive/cas_common_chemistry_lookup")
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = DRIVE_DIR / "cas_common_chemistry_results.csv"

if OUTPUT_PATH.exists():
    print(f"Found existing results already in Drive: {OUTPUT_PATH}")
else:
    print("No results file in Drive yet.")
    print("If you have a partial results CSV saved locally (e.g. a previous")
    print("download), upload it now to seed Drive with your existing progress.")
    print("Otherwise cancel/skip the upload dialog to start fresh.")
    uploaded_prev = files.upload()
    if uploaded_prev:
        local_name = next(iter(uploaded_prev))
        # a plain rename/replace fails here with "Invalid cross-device link":
        # Drive is a separate (FUSE-mounted) filesystem from local disk, and an
        # atomic rename can't cross that boundary -- copy the bytes instead.
        shutil.copy2(local_name, OUTPUT_PATH)
        Path(local_name).unlink()
        print(f"Seeded {OUTPUT_PATH} from {local_name}")
    else:
        print("No file uploaded -- starting fresh.")

print(f"\nOUTPUT_PATH is now: {OUTPUT_PATH}")

## 5. Batch lookup (resumable, checkpointed)

Writes to `cas_common_chemistry_results.csv` every 50 rows. If this cell
is interrupted (Colab disconnect, error, you stop it), just re-run it --
CAS numbers already in the output file are skipped.

In [ ]:
# OUTPUT_PATH is set in the "Resume safely across runtime restarts" cell above
REQUESTS_PER_SECOND = 3          # conservative default -- raise once you've confirmed no 429s
CHECKPOINT_EVERY = 50
MAX_RETRIES_ON_429 = 5

def load_existing_results():
    if OUTPUT_PATH.exists():
        df = pd.read_csv(OUTPUT_PATH, dtype=str)
        return df, set(df["cas_number_queried"])
    return pd.DataFrame(), set()

results_df, already_done = load_existing_results()
todo = [c for c in cas_numbers if c not in already_done]
print(f"{len(already_done)} already fetched, {len(todo)} remaining")

buffer = []
delay = 1.0 / REQUESTS_PER_SECOND

for i, cas_rn in enumerate(todo, start=1):
    for attempt in range(MAX_RETRIES_ON_429 + 1):
        status, body = fetch_detail_raw(cas_rn)
        if status == 429:
            wait = int(SESSION.headers.get("Retry-After", 5)) if attempt < MAX_RETRIES_ON_429 else 0
            wait = max(wait, 2 ** attempt)  # exponential backoff if no Retry-After header
            print(f"  [{cas_rn}] rate-limited (429), waiting {wait}s (attempt {attempt+1})")
            time.sleep(wait)
            continue
        break

    buffer.append(extract_fields(cas_rn, status, body))
    time.sleep(delay)

    if i % 10 == 0 or i == len(todo):
        print(f"[{i}/{len(todo)}] {cas_rn} -> status {status}")

    if len(buffer) >= CHECKPOINT_EVERY or i == len(todo):
        chunk = pd.DataFrame(buffer)
        results_df = pd.concat([results_df, chunk], ignore_index=True)
        results_df.to_csv(OUTPUT_PATH, index=False)
        buffer = []

print(f"\nDone. {len(results_df)} total rows in {OUTPUT_PATH}")


## 6. Summary and download

In [ ]:
import pandas as pd

final_df = pd.read_csv(OUTPUT_PATH, dtype=str)
n_found = (final_df["found"] == "True").sum()
n_not_found = (final_df["found"] == "False").sum()
print(f"{len(final_df)} rows total: {n_found} found, {n_not_found} not found in Common Chemistry")

from google.colab import files
files.download(str(OUTPUT_PATH))
